# Gene copy number plots

Boxplots of ecDNA copy number by amplification type (**Fig. 2c-d**) and oncogene evidence (**Suppl. Fig. 3**).

## Requirements
`py3` environment  
Supplementary Tables  

## TODO
- add the feature table to the supplementary tables.
- 2d: stratify by genomic region in addition to tumor type.
- SF3: stratify by tumor type.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager
import seaborn as sns
from matplotlib.offsetbox import AnnotationBbox, HPacker, TextArea
from matplotlib.transforms import offset_copy
import scipy.stats as stats
import statistics
from statsmodels.stats.multitest import fdrcorrection
from pathlib import Path
import sys
sys.path.append('../../src')
Path("out").mkdir(parents=True, exist_ok=True)
from data_imports import *
from seaborn_helper_functions import *
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Shared functions

In [ ]:
def savefig(basename):
    pngName = basename + ".png"
    svgName = basename + ".svg"
    plt.savefig(pngName,format='png',bbox_inches="tight")
    plt.savefig(svgName,format='svg',bbox_inches="tight")

def map_amplicon_type(feature):
    # map eg. 'BFB_1' to 'chromosomal'
    feature = feature.split('_')[0]
    d = {
        'ecDNA':'ecDNA',
        'BFB':'chromosomal',
        'Linear':'chromosomal',
        'Complex-non-cyclic':'chromosomal',
        'unknown':'unclassified'
    }
    if feature in d.keys():
        return d[feature]
    else:
        return feature

def import_oncogenes(file='../../data/oncogenes/oncogene_list.txt'):
    with open(file, "r") as f:
        return set(map(str.strip,f.readlines()))
def import_genes_of_interest(file='genes.txt'):
    with open(file, "r") as f:
        return list(map(str.strip,f.readlines()))


In [ ]:
genes = import_genes()
genes['classification'] = genes.feature.map(map_amplicon_type)
biosamples = import_biosamples()

# Gene copy number by amplification type
### plot_oncogene_cn_by_amplicon_class  
Produces a boxplot comparing CN between ecDNA and chromosomal amps for genes in `genes.txt`.
Strata are included which have at least 1 ecDNA and at least 1 chromosomal amp of the same tumor type; and genes are testable with at least n=3 examples in each:  
MYCN, PDGFRA, KIT, EGFR, MYC, PVT1, MET, CCND2, SREBF1, TOP3A, CCNE1  
See **Fig. 2c**.
### plot_all_cn_by_amplicon_class 
Produces a violinplot comparing CN between ecDNA and chromosomal amps for all amplicons (deduplicated) in the dataset.  
See **Fig. 2d**.
```
gene 	n_ecDNA 	n_chromosomal 	n_strata 	fold_change 	p 	p_adj
All 	269 	 	247 	 	 	22 	 	 	1.945978 	0.0001 	0.0001
```
NB. that Fig. 2d does not control for genome region amplified; so this analysis cannot distinguish between higher copy of the same or different regions within the same tumor type.
### Sensitivity analysis
2d by molecular subtype:
```
gene 	n_ecDNA 	n_chromosomal 	n_strata 	fold_change 	p 	p_adj
All 	254 	 	215 	 	 	40  		1.769115 	0.0001 	0.0001
```

In [ ]:
def add_molecular_subtype(biosamples):
    # cancer_type.cancer_subclass where a subclass exists, as in the gex model
    return biosamples.assign(molecular_subtype=biosamples.cancer_type.where(
        biosamples.cancer_subclass.isna(),
        biosamples.cancer_type + '.' + biosamples.cancer_subclass.astype(str)
    ))

def get_cluster_feature_cn(biosamples, strata='cancer_type',
                           path='../../data/source/AmpliconClassifier/pedpancan_feature_basic_properties.tsv'):
    '''
    Fig. 2d: one copy number per cluster (patient x stratum) and amplification class, the median of AmpliconClassifier's
    median_feature_CN over the cluster's ecDNA or chromosomal features. Includes features without genes, and borderline
    (LowCN) features, which S.T.2-3 count as amplified. Same columns as get_cluster_cn, with gene = 'All', so the
    Fig. 2c test and plot functions apply.
    '''
    biosamples = add_molecular_subtype(biosamples)
    df = pd.read_csv(path, sep='\t')
    df = pd.concat([df, df.feature_ID.str.extract(r'^(?P<sample_name>.+)_(?P<amplicon_number>amplicon\d+)_(?P<feature>.+)$')], axis=1)
    df['classification'] = df.feature.map(map_amplicon_type)
    df = df[df.sample_name.isin(biosamples.index) & df.classification.isin(['ecDNA','chromosomal'])]
    df = df.join(biosamples[['patient_id',strata]], on='sample_name')
    df = df.groupby(['patient_id',strata,'classification'], as_index=False).agg(
        gene_cn=('median_feature_CN','median'), # named as in get_cluster_cn
        n_biosamples=('sample_name','nunique'),
        n_features=('feature_ID','size')
    )
    df['gene'] = 'All'
    return df

def get_cluster_cn(genes, biosamples, gene_list, strata='cancer_type'):
    '''
    One copy number per gene, cluster (patient x stratum) and amplification class:
    the median gene_cn across the cluster's biosamples. Keeps all biosamples (no in_unique_tumor_set filter)
    and truncated genes (their CN matches full genes on the same amplicon).
    strata: 'cancer_type' or 'molecular_subtype' (cancer_type.cancer_subclass where a subclass exists, as in the gex model).
    '''
    biosamples = add_molecular_subtype(biosamples)
    df = genes[
        genes.gene.isin(gene_list) &
        genes.gene_cn.notna() &
        genes.classification.isin(['ecDNA','chromosomal'])
    ]
    df = df.join(biosamples[['patient_id',strata]], on='sample_name')
    # one value per biosample (rarely, a gene is on two amplicons of the same class)
    df = df.groupby(['patient_id',strata,'gene','classification','sample_name'], as_index=False)['gene_cn'].max()
    df = df.groupby(['patient_id',strata,'gene','classification'], as_index=False).agg(
        gene_cn=('gene_cn','median'),
        n_biosamples=('sample_name','size')
    )
    return df

def count_testable(df, strata='cancer_type', min_n=3):
    '''
    Clusters per gene and class in strata with both classes. A gene is tested if each class has at least min_n.
    '''
    informative = df.groupby(['gene',strata]).classification.transform('nunique') == 2
    counts = df[informative].groupby(['gene','classification']).size().unstack(fill_value=0)
    counts = counts.reindex(df.gene.unique(), fill_value=0)
    counts['n_strata'] = df[informative].groupby('gene')[strata].nunique()
    counts['n_strata'] = counts['n_strata'].fillna(0).astype(int)
    counts['tested'] = (counts.ecDNA >= min_n) & (counts.chromosomal >= min_n)
    return counts

In [ ]:
E='ecDNA'
C='chromosomal'
def setup_stratified_permutation_test_data(cn, gene, strata='cancer_type'):
    df = cn[cn.gene == gene].copy()
    # Keep strata with at least one ecDNA and one chromosomal cluster
    df = df.groupby(strata).filter(
        lambda x: x.classification.nunique() == 2
    ).reset_index(drop=True)
    # van Elteren score: rank within stratum / (stratum size + 1). Fixed under permutation.
    df['score'] = df.groupby(strata).gene_cn.rank() / (df.groupby(strata).gene_cn.transform('size') + 1)
    return df

def compute_statistic(df):
    # van Elteren statistic: sum of ecDNA scores
    return df.loc[df.classification == E,'score'].sum()

def permute(df, strata='cancer_type'):
    dfp = df.copy()
    # clusters with both classes: swap labels with 50% probability
    clusters = dfp.groupby(["patient_id",strata])
    paired = clusters.classification.transform("size") == 2
    swap = paired & (np.random.rand(clusters.ngroups) < 0.5)[clusters.ngroup()]
    dfp.loc[swap,'classification'] = dfp.loc[swap,'classification'].map({E:C, C:E})
    # clusters with one class: shuffle within stratum
    for stratum, idx in dfp[~paired].groupby(strata).groups.items():
        idx = np.array(list(idx)) # index of single-class clusters in this stratum
        dfp.loc[idx,'classification'] = np.random.permutation(
            dfp.loc[idx,'classification'].values
        )
    return dfp

def permutation_test(df, strata='cancer_type', N=10000, verbose=False):
    # one-sided: ecDNA greater
    T_perm = np.empty(N)
    for i in range(N):
        if verbose and i%1000==0:
            print("Iteration:",i)
        T_perm[i] = compute_statistic(permute(df, strata))
    T_obs = compute_statistic(df)
    p = (1 + np.sum(T_perm >= T_obs)) / (N + 1)
    return p

def hodges_lehmann(df, strata='cancer_type'):
    # stratified Hodges-Lehmann: median log copy number difference over ecDNA-chromosomal pairs from the same stratum, as a fold change
    diffs = []
    for stratum, d in df.groupby(strata):
        e = np.log(d.loc[d.classification == E,'gene_cn'].values)
        c = np.log(d.loc[d.classification == C,'gene_cn'].values)
        diffs.append(np.subtract.outer(e, c).ravel())
    return np.exp(np.median(np.concatenate(diffs)))

def run_permutation_tests(cn, strata='cancer_type', min_n=3, N=10000):
    counts = count_testable(cn, strata, min_n)
    results = []
    for gene in counts[counts.tested].index:
        df = setup_stratified_permutation_test_data(cn, gene, strata)
        results.append({
            'gene': gene,
            'n_ecDNA': (df.classification == E).sum(),
            'n_chromosomal': (df.classification == C).sum(),
            'n_strata': df[strata].nunique(),
            'fold_change': hodges_lehmann(df, strata),
            'p': permutation_test(df, strata, N)
        })
    results = pd.DataFrame(results)
    _, results['p_adj'] = fdrcorrection(results.p)
    return results

In [ ]:
def plot_cluster_cn_by_amplicon_class(cn, gene_order, results=None, angle=0, figsize=None, kind='box', italic=True):
    '''
    Boxplot of copy number by amplification class, one point per cluster (patient x stratum).
    gene_order: genes to plot, in x-axis order; genes without ecDNA or chromosomal amplifications are skipped.
    results: output of run_permutation_tests (optional). Genes in results are labelled with the BH-adjusted
        significance of the stratified test; other genes are plotted without a test.
    kind: 'box' (boxes and points) or 'violin' (split violin, ecDNA left and chromosomal right, with quartiles).
    italic: italicize the x labels (gene names).
    '''
    stars = lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    order = [g for g in gene_order if g in set(cn.gene)]
    df = cn[cn.gene.isin(order)]
    if figsize is None:
        figsize = (0.35*len(order) + 0.7, 2)
    set_plot_defaults()
    fig, ax = plt.subplots(figsize=figsize)
    linewidth=0.75
    palette = {E:'red', C:'magenta'}
    if kind == 'box':
        sns.boxplot(data=df,x='gene',y='gene_cn',hue='classification',order=order,hue_order=[E,C],
                    fliersize=0,linewidth=linewidth,palette=palette,ax=ax)
        sns.stripplot(data=df,x='gene',y='gene_cn',hue='classification',order=order,hue_order=[E,C],
                      dodge=True,ax=ax,linewidth=linewidth,size=3,legend=False,palette=palette)
    elif kind == 'violin':
        # density estimated on the log scale, cut at the data range
        sns.violinplot(data=df,x='gene',y='gene_cn',hue='classification',order=order,hue_order=[E,C],
                       split=True,inner='box',cut=0,density_norm='width',log_scale=True,
                       linewidth=linewidth,palette=palette,ax=ax)
    sns.despine()
    ax.set_yscale("log")
    ax.set_ylim(ymin=3.5,ymax=1000)
    ax.set_xlabel("")
    ax.set_ylabel("Estimated copy number")
    ax.tick_params(axis='y', left=True)
    for label in ax.get_xticklabels():
        label.set_fontstyle('italic' if italic else 'normal')
        if angle:
            label.set_rotation(angle)
            label.set_ha('center')
            label.set_va('center')
    if angle:
        # center-aligned rotated labels straddle their anchor: pad by half the tallest label so they clear the axis
        fig.canvas.draw()
        label_height = max(l.get_window_extent(fig.canvas.get_renderer()).height for l in ax.get_xticklabels()) * 72 / fig.dpi
        ax.tick_params(axis='x', pad=label_height/2 + 2)
    # test result: bracket over the two boxes, just above the highest point of the gene
    top = ax.get_xaxis_transform() # x in data, y in axes coordinates
    if results is not None:
        ymax = df.groupby('gene').gene_cn.max()
        for i, gene in enumerate(order):
            if gene in set(results.gene):
                y = ymax[gene] * 1.4
                ax.plot([i - 0.2, i + 0.2], [y, y], color='black', linewidth=linewidth, clip_on=False, scalex=False, scaley=False)
                ax.text(i, y, stars(results.set_index('gene').loc[gene,'p_adj']), ha='center', va='bottom', clip_on=False)
    # n clusters in two rows below the gene names, ecDNA then chromosomal; row labels "n (ec)", "n (chr)" with italic n
    fig.canvas.draw()
    label_bottom = min(l.get_window_extent(fig.canvas.get_renderer()).y0 for l in ax.get_xticklabels())
    y = ax.transAxes.inverted().transform((0, label_bottom))[1] - 0.03
    nobs = df.groupby(['gene','classification']).size().unstack(fill_value=0).reindex(columns=[E,C], fill_value=0)
    line = 1.3 * plt.rcParams['font.size'] # row spacing, points
    for row, (cls, name) in enumerate([(E,'ec'), (C,'chr')]):
        for i, gene in enumerate(order):
            ax.text(i, y, nobs.loc[gene,cls], ha='center', va='top',
                    transform=offset_copy(top, fig=fig, y=-row*line, units='points'))
        row_label = HPacker(children=[
            TextArea('n', textprops=dict(fontstyle='italic')),
            TextArea(f' ({name})')
        ], align='baseline', pad=0, sep=0)
        ax.add_artist(AnnotationBbox(row_label, (0, y), box_alignment=(1, 1), frameon=False, pad=0, annotation_clip=False,
                                     xycoords=offset_copy(ax.transAxes, fig=fig, y=-row*line, units='points')))
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=labels, frameon=False, loc='upper left', bbox_to_anchor=(1, 1))
    return ax

In [ ]:
# Get genes of interest, identify testable (n>=3 for ec, chr)
gene_list = [g for g in import_genes_of_interest() if g]
cn = get_cluster_cn(genes, biosamples, gene_list)
testable = count_testable(cn)
testable[testable.tested]

# run permutation tests, k=10,000
results = run_permutation_tests(cn)
results

In [ ]:
# full panel, tests on every testable gene
#ax = plot_cluster_cn_by_amplicon_class(cn, gene_list, results, angle=45, figsize=(8.6,2))
ax = plot_cluster_cn_by_amplicon_class(cn, gene_list, results, angle=45, figsize=(7.3,2))
savefig('out/oncogene_cn_expanded')

In [ ]:
# plot and test a subset only (BH across the subset)
oncogenes = ['MYCN','PDGFRA','MYC','MET','CCND2','SREBF1','EGFR','CCNE1']
results_subset = run_permutation_tests(cn[cn.gene.isin(oncogenes)])

In [ ]:
ax = plot_cluster_cn_by_amplicon_class(cn, oncogenes, results_subset, angle=45)
savefig('out/oncogene_cn_oncosubset')

In [ ]:
cn_all = get_cluster_feature_cn(biosamples)
np.random.seed(47)
results_all = run_permutation_tests(cn_all)

In [ ]:
ax = plot_cluster_cn_by_amplicon_class(cn_all, ['All'], results_all, kind='violin', italic=False, figsize=(0.75,2))
savefig('out/all_cn')
results_all

In [ ]:
# Molecular subtype
# Get genes of interest, identify testable (n>=3 for ec, chr)
gene_list = [g for g in import_genes_of_interest() if g]
cnm = get_cluster_cn(genes, biosamples, gene_list, strata='molecular_subtype')
testable = count_testable(cnm,strata='molecular_subtype')
testable[testable.tested]

# run permutation tests, k=10,000
r2 = run_permutation_tests(cnm,strata='molecular_subtype')
r2

In [ ]:
ax = plot_cluster_cn_by_amplicon_class(cnm, gene_list, r2, angle=45, figsize=(7.3,2))


In [ ]:
cnm_all = get_cluster_feature_cn(biosamples, strata='molecular_subtype')
np.random.seed(47)
r3 = run_permutation_tests(cnm_all,strata='molecular_subtype')

In [ ]:
ax = plot_cluster_cn_by_amplicon_class(cnm_all, ['All'], r3, kind='violin', italic=False, figsize=(1,2))
r3

In [ ]:
## Archive

In [ ]:
def get_interesting_genes(genes,biosamples):
    unique_tumors = set(biosamples[biosamples.in_unique_tumor_set].index)
    genes = genes[genes.sample_name.isin(unique_tumors)]
    genes = genes[genes.gene_cn.notna()]
    genes = genes[genes.is_canonical_oncogene]
    genes = genes[genes.classification.isin(['ecDNA','chromosomal'])]
    # Group by 'gene' and 'classification', then count the number of occurrences
    gene_classification_counts = genes.groupby(['gene', 'classification']).size().unstack(fill_value=0)

    # Filter genes where each classification has at least 3 occurrences
    filtered_genes = gene_classification_counts[gene_classification_counts.ge(3).all(axis=1)].index

    return(filtered_genes)

def run_statistics(df):
    # Initialize a list to store p-values
    genes = []
    p_values = []
    deltas = []

    # Perform Mann-Whitney U test for each gene
    for gene, group in df.groupby('gene'):
        genes.append(gene)
        class1_values = (group[group['classification'] == 'ecDNA']['gene_cn']).values.astype(float)
        class2_values = (group[group['classification'] == 'chromosomal']['gene_cn']).values.astype(float)
        if len(class1_values) > 1 and len(class2_values) > 1:  # Ensure there are enough values for the test
            stat, p_value = stats.mannwhitneyu(class1_values, class2_values, alternative='greater')
            p_values.append(p_value)
            deltas.append((stat*2)/(len(class1_values)*len(class2_values)) - 1)
        else:
            p_values.append(1.0)  # If there's not enough data, assign a non-significant p-value
            deltas.append(0)
    
    # Apply Benjamini-Hochberg correction
    _, p_values_corrected = fdrcorrection(p_values)
    
    # Display the corrected p-values
    for gene, p_uncor, p_corr, delta in zip(genes, p_values, p_values_corrected, deltas):
        print(f"Gene: {gene}, p-value: {p_uncor}, Corrected p-value: {p_corr}, Cliff's delta: {delta}")
    return

def add_n(ax,genes):
    # TODO: plot these counts instead of printing
    nobs = genes[['gene','classification']].value_counts()
    print(nobs)  # too much pain to add n to this plot programmatically so we'll do it post hoc in illustrator
    return nobs

def plot_oncogene_cn_by_amplicon_class(oncogenes,biosamples=None,genes=None):
    '''
    oncogenes: list of strings. eg. ['MYC'].
    If genes is provided, should be a dataframe with the columns:
        gene, gene_cn, amplicon_type 
    '''
    if biosamples is None:
        biosamples = import_biosamples()
    if genes is None:
        genes = import_genes()
        genes['classification'] = genes.feature.map(map_amplicon_type)
    genes = genes.copy()
    # get data subset of interest
    unique_tumors = set(biosamples[biosamples.in_unique_tumor_set].index)
    genes = genes[genes.sample_name.isin(unique_tumors)]
    genes = genes[genes.gene_cn.notna()]
    genes = genes[genes.gene.isin(oncogenes)]
    genes = genes[genes.classification.isin(['ecDNA','chromosomal'])]

    # Stats
    run_statistics(genes)

    # Plot
    linewidth=0.75
    set_plot_defaults()
    palette = ['red','magenta']
    #ax = sns.violinplot(data=genes, x="gene", y="gene_cn", hue="classification", inner="point", density_norm="width")
    ax = sns.boxplot(data=genes,x='gene',y='gene_cn',hue='classification',fliersize=0,linewidth=linewidth,palette=palette)
    sns.stripplot(data=genes,x='gene',y='gene_cn',hue='classification',dodge=True,ax=ax,linewidth=linewidth,legend=False,palette=palette)
    sns.despine()
    ax.set_yscale("log")
    ax.set_xlabel("")
    for label in ax.get_xticklabels():
        label.set_fontstyle('italic')
    ax.set_ylabel("Estimated copy number")
    ax.tick_params(axis='y', left=True) 
    ax.set_ylim(ymin=4,ymax=1000)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=labels,frameon=False)
    add_n(ax,genes)
    return ax

In [ ]:
#oncogenes=get_interesting_genes(genes,biosamples)
#print(oncogenes)
oncogenes=['MET','CCND2','CCNE1','PDGFRA','SREBF1','EGFR','MYC','MYCN'] # deduplicate, multiple oncogenes on the same amps

ax = plot_oncogene_cn_by_amplicon_class(oncogenes,biosamples,genes)
savefig('out/oncogene_cn')

In [ ]:
def plot_all_cn_by_amplicon_class(biosamples=None,genes=None):
    '''
    genes: list of strings. eg. ['MYC'].
    If genes is provided, should be a dataframe with the columns:
        gene, gene_cn, amplicon_type
    '''
    if biosamples is None:
        biosamples = import_biosamples()
    if genes is None:
        genes = import_genes()
        genes['classification'] = genes.feature.map(map_amplicon_type)
    genes = genes.copy()
    # get data subset of interest
    unique_tumors = set(biosamples[biosamples.in_unique_tumor_set].index)
    genes = genes[genes.sample_name.isin(unique_tumors)]
    genes = genes[genes.gene_cn.notna()]
    genes = genes[genes.classification.isin(['ecDNA','chromosomal'])]
    genes = genes.groupby(['sample_name','amplicon_number','classification'],as_index=False)['gene_cn'].max()

    # Stats
    class1_values = (genes[genes['classification'] == 'ecDNA']['gene_cn']).values.astype(float)
    class2_values = (genes[genes['classification'] == 'chromosomal']['gene_cn']).values.astype(float)
    stat, p_value = stats.mannwhitneyu(class1_values, class2_values, alternative='greater')
    c_delta = (stat*2)/(len(class1_values)*len(class2_values)) - 1
    print(f'All. median ecDNA: {statistics.median(class1_values)}. median chromosomal: {statistics.median(class2_values)}. U: {stat}. p-value: {p_value}. Cliff\'s delta: {c_delta}')
    
    # Plot
    genes['gene'] = 'All' # add dummy variable to match plot formatting of (c)
    genes['gene_cn'] = genes['gene_cn'].map(np.log10) # kernel estimates don't work on log-scaled axes
    linewidth=0.75
    set_plot_defaults(figsize=(1,2))
    palette = ['red','magenta']
    ax = sns.violinplot(data=genes, x="gene", y="gene_cn", hue="classification", hue_order=['ecDNA','chromosomal'],inner="box",
                        density_norm="width", palette=palette, linewidth=linewidth, legend=False)
    #ax = sns.boxplot(data=genes,x='gene',y='gene_cn',hue='classification',hue_order=['ecDNA','chromosomal'],fliersize=0,palette=palette)
    #sns.stripplot(data=genes,x='gene',y='gene_cn',hue='classification',hue_order=['ecDNA','chromosomal'],dodge=True,ax=ax,linewidth=1,legend=False,palette=palette)
    sns.despine()
    #ax.set_yscale("log")
    ax.set_xlabel("")
    ax.set_ylabel("Estimated copy number")
    ax.tick_params(axis='y', left=True) 
    ax.set_ylim(ymin=np.log10(4),ymax=np.log10(1000))
    #handles, labels = ax.get_legend_handles_labels()
    #ax.legend(handles=handles, labels=labels)
    add_n(ax,genes)
    return ax

In [ ]:
ax = plot_all_cn_by_amplicon_class(biosamples,genes)
savefig('out/all_cn')

# Gene copy number by oncogene status
Produces a boxplot comparing CN for all genes in Fig. 2a-b.  
**Suppl. Fig. 3**.
```
MannwhitneyuResult(statistic=np.float64(138.0), pvalue=np.float64(0.23307825211028987))
```

In [ ]:
def run_more_statistics(df):
    # Drop correlated observations
    drop = ['PVT1','BIRC2','BIRC3','DDIT3']
    df = df[~df.gene.isin(drop)].copy()
    class1_values=df[ df.is_canonical_oncogene].groupby("gene",observed=True)["gene_cn"].median()
    class2_values=df[~df.is_canonical_oncogene].groupby("gene",observed=True)["gene_cn"].median()
    stat, p_value = stats.mannwhitneyu(
        class1_values,class2_values,alternative="two-sided"
    )
    c_delta = (stat*2)/(len(class1_values)*len(class2_values)) - 1
    print(f'All. median oncogene: {statistics.median(class1_values)}. median non-oncogene: {statistics.median(class2_values)}. U: {stat}. p-value: {p_value}. Cliff\'s delta: {c_delta}')
    return df

def plot_gene_cn_by_oncogene_status(goi,biosamples=None,genes=None):
    '''
    goi: genes of interest. list of strings. eg. ['MYC'].
    If genes is provided, should be a dataframe with the columns:
        gene, gene_cn, amplicon_type 
    '''
    if biosamples is None:
        biosamples = import_biosamples()
    if genes is None:
        genes = import_genes()
        genes['classification'] = genes.feature.map(map_amplicon_type)
    genes = genes.copy()
    # get data subset of interest
    unique_tumors = set(biosamples[biosamples.in_unique_tumor_set].index)
    genes = genes[
        genes.sample_name.isin(unique_tumors) &
        genes.gene_cn.notna() &
        genes.gene.isin(goi) &
        #genes.classification.isin(['ecDNA','chromosomal'])
        (genes.classification == 'ecDNA')
    ]
    genes['gene'] = pd.Categorical(genes.gene, categories=goi, ordered=True)
    genes = genes.sort_values('gene')

    #run_more_statistics(genes)
    
    set_plot_defaults()
    ax = sns.boxplot(data=genes,x='gene',y='gene_cn',hue='is_canonical_oncogene',fliersize=0,)
    sns.stripplot(data=genes,x='gene',y='gene_cn',hue='is_canonical_oncogene',dodge=False,ax=ax,legend=False,linewidth=0.5,size=3)
    sns.despine()
    ax.set_yscale("log")
    ax.set_xlabel("")
    ax.set_ylabel("Estimated copy number")
    ax.tick_params(axis='y', left=True)
    ax.tick_params(axis="x", bottom=True)
    ax.set_ylim(ymin=4,ymax=1000)
    for label in ax.get_xticklabels():
        label.set_fontstyle('italic')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    #handles, labels = ax.get_legend_handles_labels()
    #ax.legend(handles=handles, labels=labels)
    ax.legend_.remove()
    #add_n(ax,genes)
    return ax

In [ ]:
genes = import_genes()
genes['classification'] = genes.feature.map(map_amplicon_type)
biosamples = import_biosamples()
oncogenes = import_oncogenes()
genes_of_interest = import_genes_of_interest()
ax = plot_gene_cn_by_oncogene_status(genes_of_interest,biosamples,genes)
savefig('out/gene_cn')

In [ ]:
def plot_median_cn_by_oncogene_status(goi,biosamples=None,genes=None):
    '''
    goi: genes of interest. list of strings. eg. ['MYC'].
    If genes is provided, should be a dataframe with the columns:
        gene, gene_cn, amplicon_type 
    '''
    if biosamples is None:
        biosamples = import_biosamples()
    if genes is None:
        genes = import_genes()
        genes['classification'] = genes.feature.map(map_amplicon_type)
    genes = genes.copy()
    # get data subset of interest
    unique_tumors = set(biosamples[biosamples.in_unique_tumor_set].index)
    genes = genes[
        genes.sample_name.isin(unique_tumors) &
        genes.gene_cn.notna() &
        genes.gene.isin(goi) &
        #genes.classification.isin(['ecDNA','chromosomal'])
        (genes.classification == 'ecDNA')
    ]
    genes['gene'] = pd.Categorical(genes.gene, categories=goi, ordered=True)

    genes = run_more_statistics(genes)
    genes = genes.groupby(["gene","is_canonical_oncogene"],observed=True,as_index=False)["gene_cn"].median()

    set_plot_defaults(figsize=(1,2))
    ax = sns.boxplot(data=genes,x='is_canonical_oncogene',hue='is_canonical_oncogene',order=[True,False],y='gene_cn',fliersize=0,)
    sns.stripplot(data=genes,x='is_canonical_oncogene',hue='is_canonical_oncogene',y='gene_cn',dodge=False,ax=ax,legend=False,linewidth=0.75)
    sns.despine()
    ax.set_yscale("log")
    ax.set_xlabel("")
    ax.set_ylabel("Estimated copy number")
    ax.tick_params(axis='y', left=True)
    ax.tick_params(axis="x", bottom=True)
    ax.set_ylim(ymin=4,ymax=100)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    #handles, labels = ax.get_legend_handles_labels()
    #ax.legend(handles=handles, labels=labels)
    ax.legend_.remove()
    return genes

In [ ]:
ax = plot_median_cn_by_oncogene_status(genes_of_interest,biosamples,genes)
savefig('out/median_gene_cn')